# Two-group RSA: tone-category structure by learning stage

Group-level RSA on FLT2's per-subject, per-stage RDMs (`rsa_roi.py --stage`: crossnobis on GLMsingle single-trial betas; fMRIPrep 25.2.5, T1w). Cross-validation splits each run's repetitions of a stimulus into two halves (`--cv=rep`, FLT2 `trial_splits.py`), not runs: learners change from run to run, and a run split would treat that learning as noise.. Learners (non-Mandarin) vs. native listeners (Mandarin). `09_integration/TWO_GROUP_MANUSCRIPT_PLAN.md`, optional RSA section; structure follows SSP's `GLMsingle_rsa-group.ipynb`, statistics follow `group_stats.py` so they match the univariate and PR results.

Each subject × stage × ROI RDM is compared with two categorical model RDMs (Kendall's tau-a):
- **tone**: same tone = similar. The category the learners learn.
- **talker**: same talker = similar. A non-learned acoustic control.

Predictions (plan §0): category knowledge → natives' tone fit is present from the early stage; learning → learners' tone fit rises across stages (positive trend), so the groups differ early and converge. Talker fit shouldn't follow that pattern.

Distances are cross-validated Euclidean unless FLT2 ran `--noise=shrinkage` (set `rdm_variant` below). **Before trusting anything here:** FLT2's striatal masking (`--atlas=tian-S2`) and the per-stage RDMs exist for every participant (section 1).

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join('..', '06_univariate'))
import group_stats as gs
import multivariate_data as mvd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)

bidsroot = '/ix1/bchandrasekaran/krs228/data/FLT/data_denoised'
rdm_variant = 'crossnobis_cv-rep'   # or 'crossnobis-noiseshrinkage_cv-rep'; '_cv-run' only for the check in section 1
rdm_dir = os.path.join(mvd.DERIV_DIR, mvd.GLMSINGLE_DIRNAME, f'rsa-roi_glmsingle_rdmcalc-{rdm_variant}')
out_dir = os.path.join(mvd.DERIV_DIR, mvd.GLMSINGLE_DIRNAME, 'stats_two-group', f'rsa_{rdm_variant}')
os.makedirs(out_dir, exist_ok=True)

compare_method = 'tau-a'
n_perms, n_boot, seed = 10000, 10000, 0   # matching robustness_checks.py
colors = {'non-Mandarin': '#2a6fdb', 'Mandarin': '#d9822b'}   # as in group_level_two_group.ipynb
labels = {'non-Mandarin': 'learners', 'Mandarin': 'natives'}

## 1. Load and check

In [ ]:
participants = mvd.load_participants(bidsroot)
rdms = mvd.load_subject_rdms(rdm_dir, participants)
fits = mvd.add_composites(mvd.model_fits(rdms, participants, method=compare_method))

have = pd.DataFrame(list(rdms), columns=['participant_id', 'stage'])
print('missing a stage:', sorted(set(participants.participant_id) - set(
    have.groupby('participant_id').stage.nunique().loc[lambda n: n == 3].index)))
print('ROIs:', sorted(fits.region_hemi.unique()))
gs.coverage(fits)

### Check: repetition vs. run split, natives

Natives shouldn't change with learning, so their tone fits from the repetition split and the run split (`rsa_roi.py --cv=run`) should agree. A systematic difference points to noise shared within runs, which the repetition split can't remove. Skipped if the run-split RDMs weren't computed.

In [ ]:
run_dir = rdm_dir.replace('_cv-rep', '_cv-run')
if os.path.isdir(run_dir):
    natives = participants[participants.participant_group == 'Mandarin']
    by_split = pd.concat([mvd.model_fits(mvd.load_subject_rdms(d, natives), natives, method=compare_method).assign(cv=cv)
                          for cv, d in [('rep', rdm_dir), ('run', run_dir)]])
    wide = gs.bilateral_mean(by_split.rename(columns={'contrast': 'c'}).assign(contrast=lambda d: d.c + '_' + d.cv))
    wide = wide.pivot_table(index=['participant_id', 'region', 'learning_stage'], columns='contrast', values='beta')
    print('natives, tone fit: repetition minus run split, by stage')
    print((wide['tone_fit_rep'] - wide['tone_fit_run']).groupby('learning_stage').agg(['mean', 'std']).round(3))
else:
    print('no run-split RDMs at', run_dir)

### QC: model RDMs and group-mean data RDMs

One ROI, early vs. final. Stimuli are ordered tone-major, so tone structure shows as 4 × 4 blocks on the diagonal.

In [ ]:
qc_roi = 'L-HG'
vectors, stimuli = mvd.group_mean_rdms(rdms, participants)
models = mvd.model_rdms(stimuli)
order = np.argsort(stimuli)   # tone-major

def square(vector):
    n = len(stimuli)
    m = np.zeros((n, n))
    m[np.triu_indices(n, 1)] = vector
    m = m + m.T
    return m[np.ix_(order, order)]

panels = [(f'{f} model', models[f].get_vectors()[0]) for f in models]
panels += [(f'{labels[g]} {stage}', np.mean(list(vectors[(g, stage, qc_roi)].values()), axis=0))
           for g in gs.GROUPS for stage in ('early', 'final') if (g, stage, qc_roi) in vectors]
fig, axes = plt.subplots(1, len(panels), figsize=(2.2 * len(panels), 2.4))
for ax, (title, vector) in zip(axes, panels):
    ax.imshow(square(vector), cmap='viridis')
    ax.set_title(title, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(qc_roi)
fig.tight_layout()

### Noise ceiling

Per group × stage × ROI: lower and upper bounds on the fit any model could reach given between-subject reliability. A low ceiling means that ROI's RDMs are too noisy to interpret a null fit.

In [ ]:
ceiling = mvd.noise_ceiling(rdms, participants, method=compare_method)
ceiling.to_csv(os.path.join(out_dir, 'noise_ceiling.tsv'), sep='\t', index=False)
ceiling.pivot_table(index='region_hemi', columns=['participant_group', 'learning_stage'], values='lower').round(3)

## 2. Model fits by stage and group

Hemispheres averaged per participant; mean ± SEM.

In [ ]:
bilat = gs.bilateral_mean(fits)
regions = list(mvd.COMPOSITES) + sorted(r for r in bilat.region.unique() if r not in mvd.COMPOSITES)
for factor in mvd.STIMULUS_FACTORS:
    sub = bilat[bilat.contrast == f'{factor}_fit']
    fig, axes = plt.subplots(1, len(regions), figsize=(2.2 * len(regions), 2.6), sharey=True)
    for ax, region in zip(np.atleast_1d(axes), regions):
        for group in gs.GROUPS:
            g = sub[(sub.region == region) & (sub.participant_group == group)].groupby('learning_stage').beta
            ax.errorbar(range(3), g.mean().reindex(gs.STAGE_ORDER), yerr=g.sem().reindex(gs.STAGE_ORDER),
                        marker='o', capsize=3, color=colors[group], label=labels[group])
        ax.axhline(0, color='0.6', lw=0.8)
        ax.set_xticks(range(3), gs.STAGE_ORDER)
        ax.set_title(region)
    np.atleast_1d(axes)[0].set_ylabel(f'{factor} model fit (tau-a)')
    np.atleast_1d(axes)[0].legend(frameon=False)
    fig.tight_layout()

## 3. Is there category structure? Fit vs. 0 within each group

Per stage and stages averaged; FDR across regions within group × model × network (× stage).

In [ ]:
baseline = gs.baseline_tests(fits)
baseline_by_stage = gs.baseline_tests(fits, by_stage=True)
baseline.to_csv(os.path.join(out_dir, 'fit_vs_0.tsv'), sep='\t', index=False)
baseline_by_stage.to_csv(os.path.join(out_dir, 'fit_vs_0_by-stage.tsv'), sep='\t', index=False)
baseline_by_stage[baseline_by_stage.contrast == 'tone_fit'].pivot_table(
    index=['network', 'region'], columns=['participant_group', 'learning_stage'], values='t').round(2)

## 4. Does tone structure emerge with learning? Trend, learners vs. natives

Per-participant `[-1, 0, 1]` trend (final − early) in tone fit, compared between groups. The full battery (sign-flip, LOSO, bootstrap, group-label permutation) runs on the composites and the two striatal regions of the univariate headline (aCAU, aPUT); every region gets the Welch/permutation test with FDR. Talker fit alongside as the control -- but the fits aren't independent: tau-a is rank-based, so as tone structure grows, the same RDM ranks talker pairs lower and talker fit falls even if talker information is unchanged (seen in synthetic data). Read a talker-fit decline next to a tone-fit rise as a shift in relative structure, not lost talker information; a model fitting both at once (e.g. rsatoolbox `ModelWeighted` with tone and talker components) would separate them. Exploratory: no RSA test was pre-specified.

In [ ]:
trends = gs.subject_trends(fits)
focus = [r for r in list(mvd.COMPOSITES) + ['aCAU', 'aPUT'] if r in set(trends.region)]
battery = pd.concat([gs.trend_group_difference(trends, f'{factor}_fit', region,
                                               n_perms=n_perms, n_boot=n_boot, seed=seed)
                     for factor in mvd.STIMULUS_FACTORS for region in focus], ignore_index=True)
battery.to_csv(os.path.join(out_dir, 'fit_trend_focus-regions.tsv'), sep='\t', index=False)
battery.round(3)

In [ ]:
per_region = fits[~fits.region.isin(mvd.COMPOSITES)]
trend_diff = gs.trend_difference_tests(gs.subject_trends(per_region),
                                       contrasts=[f'{f}_fit' for f in mvd.STIMULUS_FACTORS],
                                       n_perms=n_perms, seed=seed)
trend_diff.to_csv(os.path.join(out_dir, 'fit_trend_group-difference.tsv'), sep='\t', index=False)
trend_diff.round(3)

## 5. Do natives start ahead and learners catch up? Group difference per stage

Learners − natives per stage (negative = natives higher). The category-knowledge prediction is a negative difference early that shrinks by the final stage. FDR across regions within model × network × stage.

In [ ]:
stage_diff = gs.group_difference_tests(per_region, contrasts=[f'{f}_fit' for f in mvd.STIMULUS_FACTORS],
                                       by_stage=True, n_perms=n_perms, seed=seed)
stage_diff.to_csv(os.path.join(out_dir, 'fit_group-difference_by-stage.tsv'), sep='\t', index=False)
stage_diff[stage_diff.contrast == 'tone_fit'].pivot_table(
    index=['network', 'region'], columns='learning_stage', values='d')[list(gs.STAGE_ORDER)].round(2)

In [ ]:
anova = gs.group_by_stage_anova(per_region)
anova.to_csv(os.path.join(out_dir, 'fit_group-by-stage-anova.tsv'), sep='\t', index=False)
anova[anova.q < .05].round(3)

## 6. Export

Per-participant fit trends for integration with the striatal and PR results (plan R7).

In [ ]:
trends.to_csv(os.path.join(out_dir, 'fit_trend_scalars.tsv'), sep='\t', index=False)
fits.to_csv(os.path.join(out_dir, 'model_fits.tsv'), sep='\t', index=False)
print('wrote', out_dir)